# DataLoader 和 Dataset 實戰教程

本教程深入講解 PyTorch 的資料處理系統,包括如何有效地加載、處理和增強資料。

## 目錄
1. Dataset 基礎
2. DataLoader 詳解
3. 資料增強
4. 實戰案例
5. 進階技巧

**作者:** AI Learning Notes  
**最後更新:** 2025-01

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

print(f"PyTorch 版本: {torch.__version__}")

## 1. Dataset 基礎

`Dataset` 是 PyTorch 中用於組織和訪問資料的抽象類。

### 1.1 最簡單的 Dataset

In [ ]:
class SimpleDataset(Dataset):
    """最簡單的 Dataset 實現"""
    
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    
    def __len__(self):
        """返回資料集的大小"""
        return len(self.data)
    
    def __getitem__(self, idx):
        """根據索引返回一個樣本"""
        return self.data[idx], self.labels[idx]

# 創建示例數據
data = torch.randn(100, 10)
labels = torch.randint(0, 5, (100,))

# 創建 Dataset
dataset = SimpleDataset(data, labels)

print(f"Dataset 大小: {len(dataset)}")
print(f"第一個樣本: {dataset[0]}")
print(f"數據形狀: {dataset[0][0].shape}")
print(f"標籤: {dataset[0][1]}")

### 1.2 TensorDataset - 快速創建 Dataset

In [ ]:
# 使用 TensorDataset (PyTorch 內建)
data = torch.randn(100, 10)
labels = torch.randint(0, 5, (100,))

tensor_dataset = TensorDataset(data, labels)

print(f"TensorDataset 大小: {len(tensor_dataset)}")
print(f"第一個樣本: {tensor_dataset[0]}")

### 1.3 自定義圖像 Dataset

In [ ]:
class CustomImageDataset(Dataset):
    """自定義圖像 Dataset"""
    
    def __init__(self, image_paths, labels, transform=None):
        """
        Args:
            image_paths: 圖像文件路徑列表
            labels: 標籤列表
            transform: 可選的轉換操作
        """
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # 加載圖像
        image_path = self.image_paths[idx]
        # image = Image.open(image_path).convert('RGB')
        
        # 為了示例,我們創建隨機圖像
        image = torch.randn(3, 224, 224)
        
        label = self.labels[idx]
        
        # 應用轉換
        if self.transform:
            image = self.transform(image)
        
        return image, label

# 創建示例
image_paths = [f"image_{i}.jpg" for i in range(100)]
labels = torch.randint(0, 10, (100,))

dataset = CustomImageDataset(image_paths, labels)
print(f"圖像 Dataset 大小: {len(dataset)}")
print(f"第一個樣本形狀: {dataset[0][0].shape}")

### 1.4 使用 random_split 分割數據集

In [ ]:
# 創建一個數據集
full_dataset = TensorDataset(torch.randn(1000, 10), torch.randint(0, 5, (1000,)))

# 分割為訓練集和驗證集 (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"完整數據集大小: {len(full_dataset)}")
print(f"訓練集大小: {len(train_dataset)}")
print(f"驗證集大小: {len(val_dataset)}")

## 2. DataLoader 詳解

`DataLoader` 負責批次加載資料,支持多進程、打亂等功能。

### 2.1 基本使用

In [ ]:
# 創建數據集
dataset = TensorDataset(torch.randn(100, 10), torch.randint(0, 5, (100,)))

# 創建 DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=16,      # 批次大小
    shuffle=True,       # 是否打亂
    num_workers=0,      # 使用多少個進程加載數據
    drop_last=False     # 是否丟棄最後不足一個批次的數據
)

# 迭代數據
for batch_idx, (data, labels) in enumerate(dataloader):
    print(f"Batch {batch_idx + 1}:")
    print(f"  Data shape: {data.shape}")
    print(f"  Labels shape: {labels.shape}")
    if batch_idx >= 2:
        break

### 2.2 DataLoader 重要參數

In [ ]:
dataset = TensorDataset(torch.randn(100, 10), torch.randint(0, 5, (100,)))

# 完整參數示例
dataloader = DataLoader(
    dataset,
    batch_size=32,           # 批次大小
    shuffle=True,            # 每個 epoch 打亂數據
    num_workers=0,           # 多進程加載 (Windows 建議設為 0)
    pin_memory=True,         # 將數據釘在內存中 (加速 GPU 傳輸)
    drop_last=True,          # 丟棄最後不足一個批次的數據
    timeout=0,               # 從 worker 獲取數據的超時時間
    persistent_workers=False # 是否保持 worker 進程
)

print(f"DataLoader 配置:")
print(f"  Batch size: {dataloader.batch_size}")
print(f"  Num workers: {dataloader.num_workers}")
print(f"  總批次數: {len(dataloader)}")

### 2.3 自定義 collate_fn

In [ ]:
def custom_collate_fn(batch):
    """
    自定義如何將多個樣本組合成一個批次
    
    Args:
        batch: list of (data, label) tuples
    
    Returns:
        批次數據
    """
    data_list, label_list = zip(*batch)
    
    # 堆疊數據
    data = torch.stack(data_list)
    labels = torch.tensor(label_list)
    
    # 可以在這裡做額外的處理
    # 例如: 數據標準化、填充等
    
    return data, labels

# 使用自定義 collate_fn
dataloader = DataLoader(
    dataset,
    batch_size=16,
    collate_fn=custom_collate_fn
)

# 測試
data, labels = next(iter(dataloader))
print(f"Data shape: {data.shape}")
print(f"Labels shape: {labels.shape}")

## 3. 資料增強

使用 `torchvision.transforms` 進行資料增強。

### 3.1 常用的變換操作

In [ ]:
# 訓練時的變換 (包含資料增強)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),           # 隨機裁剪並縮放
    transforms.RandomHorizontalFlip(),           # 隨機水平翻轉
    transforms.RandomRotation(10),               # 隨機旋轉
    transforms.ColorJitter(                      # 顏色抖動
        brightness=0.2, 
        contrast=0.2, 
        saturation=0.2, 
        hue=0.1
    ),
    transforms.ToTensor(),                       # 轉為 Tensor
    transforms.Normalize(                        # 標準化
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 驗證/測試時的變換 (不包含隨機性)
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("訓練變換:", train_transform)
print("\n驗證變換:", val_transform)

### 3.2 自定義變換

In [ ]:
class AddGaussianNoise:
    """添加高斯噪聲"""
    
    def __init__(self, mean=0., std=0.1):
        self.mean = mean
        self.std = std
    
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean
    
    def __repr__(self):
        return f'{self.__class__.__name__}(mean={self.mean}, std={self.std})'

# 使用自定義變換
custom_transform = transforms.Compose([
    transforms.ToTensor(),
    AddGaussianNoise(mean=0, std=0.05),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

print("自定義變換:", custom_transform)

## 4. 實戰案例

完整的資料處理流程示例。

### 4.1 MNIST 數據集

In [ ]:
# 定義變換
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST 的均值和標準差
])

# 下載並加載 MNIST 數據集
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# 創建 DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

print(f"訓練集大小: {len(train_dataset)}")
print(f"測試集大小: {len(test_dataset)}")
print(f"訓練批次數: {len(train_loader)}")
print(f"測試批次數: {len(test_loader)}")

# 查看一個批次
images, labels = next(iter(train_loader))
print(f"\nBatch images shape: {images.shape}")
print(f"Batch labels shape: {labels.shape}")

### 4.2 可視化數據

In [ ]:
def imshow(img, title=None):
    """顯示圖像"""
    img = img / 2 + 0.5  # 反標準化
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title)
    plt.axis('off')

# 獲取一批圖像
dataiter = iter(train_loader)
images, labels = next(dataiter)

# 創建圖像網格
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for idx, ax in enumerate(axes.flat):
    if idx < len(images):
        ax.imshow(images[idx].squeeze(), cmap='gray')
        ax.set_title(f'Label: {labels[idx].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 4.3 完整的訓練示例

In [ ]:
# 定義簡單的 CNN 模型
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = nn.functional.relu(x)
        x = self.conv2(x)
        x = nn.functional.relu(x)
        x = nn.functional.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = nn.functional.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        return nn.functional.log_softmax(x, dim=1)

# 訓練函數
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = nn.functional.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        
        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

# 測試函數
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += nn.functional.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, '
          f'Accuracy: {correct}/{len(test_loader.dataset)} '
          f'({100. * correct / len(test_loader.dataset):.2f}%)\n')

# 訓練模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("開始訓練...")
for epoch in range(1, 3):  # 訓練 2 個 epoch
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)

print("訓練完成!")

## 5. 進階技巧

提升資料加載效率的技巧。

### 5.1 WeightedRandomSampler - 處理不平衡數據

In [ ]:
from torch.utils.data import WeightedRandomSampler

# 創建不平衡數據集
# 類別 0: 900 個樣本
# 類別 1: 100 個樣本
data = torch.randn(1000, 10)
labels = torch.cat([torch.zeros(900), torch.ones(100)]).long()

# 計算每個類別的權重
class_counts = torch.bincount(labels)
class_weights = 1. / class_counts.float()
sample_weights = class_weights[labels]

print(f"類別數量: {class_counts}")
print(f"類別權重: {class_weights}")

# 創建加權採樣器
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# 使用採樣器
dataset = TensorDataset(data, labels)
dataloader = DataLoader(
    dataset,
    batch_size=32,
    sampler=sampler  # 使用 sampler 時不能設置 shuffle
)

# 驗證採樣平衡性
sampled_labels = []
for _, labels in dataloader:
    sampled_labels.extend(labels.tolist())

sampled_counts = torch.bincount(torch.tensor(sampled_labels))
print(f"\n採樣後的類別數量: {sampled_counts}")

### 5.2 SubsetRandomSampler - 自定義採樣

In [ ]:
from torch.utils.data import SubsetRandomSampler

# 創建數據集
dataset = TensorDataset(torch.randn(1000, 10), torch.randint(0, 5, (1000,)))

# 手動分割訓練集和驗證集
indices = list(range(len(dataset)))
split = int(0.2 * len(dataset))

# 打亂索引
np.random.seed(42)
np.random.shuffle(indices)

train_indices = indices[split:]
val_indices = indices[:split]

# 創建採樣器
train_sampler = SubsetRandomSampler(train_indices)
val_sampler = SubsetRandomSampler(val_indices)

# 創建 DataLoader
train_loader = DataLoader(dataset, batch_size=32, sampler=train_sampler)
val_loader = DataLoader(dataset, batch_size=32, sampler=val_sampler)

print(f"訓練樣本數: {len(train_indices)}")
print(f"驗證樣本數: {len(val_indices)}")

### 5.3 效能優化技巧

In [ ]:
# 效能優化建議
print("DataLoader 效能優化技巧:\n")
print("1. num_workers: 設置合適的工作進程數")
print("   - CPU: 可以設置為 CPU 核心數")
print("   - Windows: 建議設置為 0 (避免多進程問題)")
print("   - Linux/Mac: 可以設置為 4-8")
print()
print("2. pin_memory: 使用釘住內存加速 GPU 傳輸")
print("   dataloader = DataLoader(..., pin_memory=True)")
print()
print("3. prefetch_factor: 每個 worker 預取的批次數")
print("   dataloader = DataLoader(..., prefetch_factor=2)")
print()
print("4. persistent_workers: 保持 worker 進程活躍")
print("   dataloader = DataLoader(..., persistent_workers=True)")
print()
print("5. 在 __getitem__ 中避免重複的計算")
print("   - 預先計算可以緩存的內容")
print("   - 使用高效的圖像加載庫 (如 pillow-simd)")
print()
print("6. 使用適當的 batch_size")
print("   - 太小: 利用不了並行計算")
print("   - 太大: 可能導致 OOM")
print("   - 建議: 從 32/64 開始調整")

## 總結

本教程涵蓋了 PyTorch 資料處理的核心內容:

1. **Dataset**: 自定義數據集、TensorDataset、圖像數據集
2. **DataLoader**: 批次加載、多進程、採樣策略
3. **資料增強**: torchvision.transforms、自定義變換
4. **實戰案例**: MNIST 完整訓練流程
5. **進階技巧**: 加權採樣、效能優化

### 最佳實踐

1. 總是使用 `DataLoader` 而不是手動批次化
2. 訓練時打亂數據 (shuffle=True)
3. 驗證/測試時不打亂 (shuffle=False)
4. 使用資料增強提升模型泛化能力
5. 根據硬件調整 num_workers 和 batch_size
6. 使用 pin_memory 加速 GPU 訓練
7. 處理不平衡數據時使用加權採樣

### 參考資源

- [PyTorch Data Loading Tutorial](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html)
- [torchvision.transforms](https://pytorch.org/vision/stable/transforms.html)
- [torch.utils.data](https://pytorch.org/docs/stable/data.html)